In [0]:
from pyspark.sql import functions as F

df = (
    spark.readStream
        .format("cloudFiles")       
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", "/Volumes/retailnew/bronze/schema_logs/marketing") 
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .load("/Volumes/retailnew/bronze/raw_data/marketing")
        .withColumn("ingest_ts", F.current_timestamp())
)

(
    df.writeStream
      .format("delta")
      .option("checkpointLocation", "/Volumes/retailnew/bronze/checkpoints/marketing/")   # CHECKPOINT HERE
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .table("retailnew.bronze.marketing")   # WRITE DIRECTLY TO UC TABLE
)